<a href="https://colab.research.google.com/github/sungyup-jung/projects/blob/main/VaR_and_ES_Validation_Engime_for_Illiquid_Private_Credit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Executive Summary
Quantifying Value-at-Risk (VaR) and Expected Shortfall (ES) on illiquid private credit portfolios presents two severe structural model risks that cause traditional Gaussian bank models to drastically understate tail capital requirements:

1. State Pricing / Volatility Unsmoothing: Private credit loans are marked quarterly or annually. These appraisal-based marks exhibit high serial autocorrelaton $(\phi > 0.5)$, artificially dampening observed variance ($\sigma^{2}_{obs} << \sigma^{2}_{true}$) and masking true portfolio tail risk.
2. Illiquidity Liquidation Horizon & Fat-Tailed Jump Dynamics: Unlike liquid high-yield bonds (which can be liquidated in 10 to 30 days), unwinding or restructuring a private debt position requires an extended Liquidity Horizon ($T_{\text{liq}} \in [90, 180]$ days) and incurs a non-linear Exogenous Liquidation Haircut. Furthermore, credit defaults are non-diffusive; they manifest as sudden Poisson jump events under Merton's structural framework.

# 2. Mathematical Framework

1. $AR(1)$ Return Unsmoothing (Geltner-Ross-Zisler Transformation)
Let $R^{\text{obs}}_{t}$ be the observed quartely return of an illiquid credit loan. Observed returns follow a first-order autoregressive process driven by the unobserved true economic return $R^{\text{true}}_{t}$:

$$R^{\text{obs}}_{t} = \phi R^{\text{obs}}_{t-1} + (1-\phi) R^{\text{true}}_{t}, \phi \in [0,1)$$

Where $\phi$ is the smoothing parameter (autocorrelation). The validator reconstructs the true economic return series and its unsmoothed volatility $\sigma^{\text{true}}$:

$$R^{\text{true}}_{t} = \frac{R^{\text{obs}}_{t} - \phi R^{\text{obs}}_{t-1}}{1-\phi}$$

$$\sigma_{\text{true}} = \sigma_{\text{obs}} \frac{\sqrt{1 - \phi^{2}}}{1-\phi}$$ (Note: if $\phi = 0.6$, $\sigma_{\text{true}} \approx 2.0 \times \sigma_{\text{obs}})$  

2. Multi-Asset Merton Jump-Diffusion Process

The firm value asset process $A_{i}(t)$ for obligor $i$ follows a structural Merton Jump-Diffusion equation incorporating continuous Gaussian drift/volatility and discrete Poisson jump events:

$$\frac{dA_{i}(t)}{A_{i}(t^{-})} = (\mu_{i} - \lambda_{i}k_{i})dt + \sigma_{i}dW_{i}(t) + (J_{i}-1)dN_{i}(t)$$

Where:
* $W_{i}(t)$: Brownian motion correlated across obligors via factor covariance matrix $\Sigma$.
* $N_{i}(t)$: Poisson process with jump intensity rate $\lambda_{i}$ (annual jump frequency).
* $J_{i}$: Log-normal jump magnitude, $ln J_{i} \sim N(\mu_{J,i}, \sigma^{2}_{J,i})$, capturing sudden credit distress or manufacturing shocks.
* $k_{i} = \mathbb{E}[J_{i}-1] = exp(\mu_{J,i} + \frac{1}{2}\sigma^{2}_{J,i})-1$: compensator ensuring martingale consistency.

Under log-returns over liquidation horizon $T_{\text{liq}}$, the terminal asset value $A_{i}(T_{\text{liq}})$ is:


$$ln A_{i}(T_{\text{liq}}) = ln A_{i}(0) + \left(\mu_{i} - \frac{1}{2}\sigma^{2}_{i} - \lambda_{i}k_{i}\right)T_{\text{liq}} + \sigma_{i}\sqrt{T_{\text{liq}}}Z_{i} + \Sigma^{N_{i}(T_{\text{liq}})}_{j=1}Y_{i,j}$$

where $Z_{i} \sim N(0,1)$ and $Y_{i,j} \sim N(\mu_{J,i}, \sigma^{2}_{J,i})$.

3. Liquidity-Adjusted Loss, L-VaR, and L-ES

If terminal asset value $A_{i}(T_{\text{liq}})$ drops below default threshold $K_{i}$, default occurs. For non-defaulted loans, mark-to-market loss is governed by asset value drawdown. Additionally, liquidating illiquid debt over horizon $T_{\text{liq}}$ incurs an exogenous liquidation haircut $\eta_{i} \cdot \sqrt{T_{\text{liq}}}$:

\begin{equation}
Loss_{i}(T_{\text{liq}}) =
\begin{cases}
EAD_{i} \times LGD_{i}, & \qquad \qquad \text{if } A_{i}(T_{\text{liq}}) < K_{i} \text{     (Default)}  \\
EAD_{i} \times \left (1 - \frac{A_{i}(T_{\text{liq}})}{A_{i}(0)} \right) + EAD_{i} \times \eta_{i} \sqrt{T_{\text{liq}}}, & \qquad \qquad \text{if } A_{i}(T_{\text{liq}}) \geq K_{i} \text{     (MTM Loss + Liquidity Friction)} \\
\end{cases}
\end{equation}
\
For portfolio loss $L = \Sigma_{i} \text{Loss}_{i}(T_{\text{liq}})$, Liquidity-Adjusted $\text{VaR}(L-\text{VaR}_{\alpha})$ and Expected Shortfall ($\text{L} - \text{ES}_{\alpha}$) at confidence level $\alpha = 97.5\%$ or $99.0\%$ are defined as:
$$\text{L}-\text{VaR}_{\alpha} = \text{inf}\{ l \in \mathbb{R}: P(L>l) \leq 1 - \alpha \}$$
\
$$\text{L}-\text{ES}_{\alpha}=\mathbb{E}[L|L \geq \text{L} - \text{VaR}_{\alpha}] = \frac{1}{1-\alpha}\int^{1}_{\alpha}\text{L}-\text{VaR}_{u}du$$




In [4]:
import numpy as np
import pandas as pd
import scipy.stats as stats

class IlliquidPrivateCreditVaRValidator:
  """
  Independent Model Risk Validation Engine for Illiquid Private Credit Portfolios.
  Evaluates Return Unsmoothing, Merton Jump-Diffusion Asset Dynamics, L-VaR, and L-ES
  """
  def __init__(self, portfolio_df: pd.DataFrame, correlation_matrix: np.ndarray):
    """
    portfolio_df columns:
    ['Obligor_ID', 'EAD', 'LGD', 'Observed_Ann_Vol', 'Smoothing_Phi',
    'Jump_Intensity_Lambda', 'Jump_Mean_MuJ', 'Jump_Std_SigmaJ',
    'Default_Barrier_K', 'Liquidity_Haircut_Eta']
    """
    self.portfolio = portfolio_df.copy()
    self.corr_matrix = correlation_matrix
    self.n_assets = len(portfolio_df)

  def unsmooth_volatility(self) -> pd.Series:
    """
    Applies Geltner AR(1) unsmoothing to adjust for stale appraisal-based marks.
    """
    obs_vol = self.portfolio['Observed_Ann_Vol']
    phi = self.portfolio['Smoothing_Phi']

    # True Volatility formula: sigma_true = sigma_obs * sqrt(1 - phi^2) / (1 - phi)
    unsmoothed_vol = obs_vol * (np.sqrt(1.0 - phi**2) / (1.0 - phi))
    return unsmoothed_vol

  def simulate_merton_jump_diffusion_losses(
      self,
      n_simulations: int = 50000,
      liquidity_horizon_days: int = 90,
      use_unsmoothed_vol: bool = True,
      enable_jumps: bool = True
  ) -> np.ndarray:
    """
    Simulates portfolio losses over the specified liquidity horizon using
    correlated Merton Jump-Diffusion processes.
    """
    T_liq = liquidity_horizon_days / 365.0

    # Select Volatility (Unsmoothed vs Raw Observed)
    if use_unsmoothed_vol:
      vols = self.unsmooth_volatility().values
    else:
      vols = self.portfolio['Observed_Ann_Vol'].values

    ead = self.portfolio['EAD'].values
    lgd = self.portfolio['LGD'].values
    lambdas = self.portfolio['Jump_Intensity_Lambda'].values if enable_jumps else np.zeros(self.n_assets)
    mu_j = self.portfolio['Jump_Mean_MuJ'].values
    sigma_j = self.portfolio['Jump_Std_SigmaJ'].values
    barriers = self.portfolio['Default_Barrier_K'].values
    eta = self.portfolio['Liquidity_Haircut_Eta'].values

    # Compensator k = exp(mu_j + 0.5 * sigma_j^2) - 1
    k_comp = np.exp(mu_j + 0.5 * sigma_j**2) - 1.0

    # Correlated Brownian Motion Shocks
    cholesky_matrix = np.linalg.cholesky(self.corr_matrix)
    uncorrelated_z = np.random.normal(0, 1, size = (n_simulations, self.n_assets))
    correlated_z = uncorrelated_z @ cholesky_matrix.T

    # Initialize Terminal Log Asset Values (A0 normalized to 1.0)
    drift = (0.02 - 0.5 * vols**2 - lambdas * k_comp) * T_liq
    diffusion = vols * np.sqrt(T_liq) * correlated_z

    log_asset_ratio = drift + diffusion

    # Add Compound Poisson Jumps
    if enable_jumps:
      for i in range(self.n_assets):
        if lambdas[i] > 0:
          # Number of jumps in T_liq for each simulation path
          n_jumps = np.random.poisson(lambdas[i] * T_liq, size=n_simulations)
          max_jumps = np.max(n_jumps)

          if max_jumps > 0:
            jump_draws = np.random.normal(mu_j[i], sigma_j[i], size=(n_simulations, max_jumps))
            # Mask jumps exceeding n_jumps count
            mask = np.arange(max_jumps) < n_jumps[:, None]
            total_jump_impact = np.sum(jump_draws * mask, axis=1)
            log_asset_ratio[:,i] += total_jump_impact

    terminal_asset_ratio = np.exp(log_asset_ratio)

    # Portfolio Loss Calculation Path-by-Path
    portfolio_losses = np.zeros(n_simulations)

    for i in range(self.n_assets):
      a_terminal = terminal_asset_ratio[:,i]

      # Default Condition: Asset value falls below default barrier K
      is_default = a_terminal < barriers[i]

      # Default Loss = EAD * LGD
      default_loss = ead[i] * lgd[i]

      # Non-Default MTM Loss + Liquidity Friction Cost
      mtm_drawdown = np.maximum(0.0, 1.0 - a_terminal)
      liquidity_friction = eta[i] * np.sqrt(T_liq)
      non_default_loss = ead[i] * (mtm_drawdown + liquidity_friction)

      # Path loss assignment
      path_loss = np.where(is_default, default_loss, non_default_loss)
      portfolio_losses += path_loss

    return portfolio_losses

  def calculate_var_es_metrics(self, loss_distribution: np.ndarray, confidence_levels: list = [0.975, 0.990]) -> dict:
    """
    Computes VaR and Expected Shortfall (ES) at specified confidence levels.
    """
    results = {}
    for alpha in confidence_levels:
      var_val = np.percentile(loss_distribution, alpha * 100.0)
      es_val = np.mean(loss_distribution[loss_distribution >= var_val])

      results[f"VaR_{alpha*100:.1f}"] = float(var_val)
      results[f"ES_{alpha*100:.1f}"] = float(es_val)

    return results

  def run_mrm_model_challenge_suite(self, n_sims: int = 50000) -> pd.DataFrame:
    """
    Executes a multi-scenario MRM independent challenge benchmarking:
    1. Baseline Flawed Front-Office Model (Observed Vol, No Jumps, 10-Day Horizon)
    2. Medium Risk Model (Unsmoothed Vol, No Jumps, 90-day Liquidity Horizon)
    3. Comprehensive Validator Champion Model (Unsmoothed Vol, Merton Jumps, 90-day L-VaR/L-ES)
    """
    # Model 1: Front-Office Flawed Benchmark (10-Day, Observed Vol, No Jumps)
    losses_fo = self.simulate_merton_jump_diffusion_losses(
        n_simulations=n_sims, liquidity_horizon_days=10, use_unsmoothed_vol=False, enable_jumps=False
    )
    metrics_fo = self.calculate_var_es_metrics(losses_fo)

    # Model 2: Intermediate Model (90-Day Horizon, Unsmoothed Vol, No Jumps)
    losses_mid = self.simulate_merton_jump_diffusion_losses(
        n_simulations=n_sims, liquidity_horizon_days=90, use_unsmoothed_vol=True, enable_jumps=False
    )
    metrics_mid = self.calculate_var_es_metrics(losses_mid)

    # Model 3: Validator Champion Model (90-Day Horizon, Unsmoothed Vol + Merton Jumps)
    losses_champ = self.simulate_merton_jump_diffusion_losses(
        n_simulations=n_sims, liquidity_horizon_days=90, use_unsmoothed_vol=True, enable_jumps=True
    )
    metrics_champ = self.calculate_var_es_metrics(losses_champ)

    total_portfolio_ead = self.portfolio['EAD'].sum()

    comparison = [
        {"Model_Specification": "1. Front-Office Baseline (10D, Smooth Vol, No Jumps)",
         "VaR_97.5_Dollar": f"${metrics_fo['VaR_97.5']:,.2f}",
         "ES_97.5_Dollar": f"${metrics_fo['ES_97.5']:,.2f}",
         "ES_99.0_Dollar": f"${metrics_fo['ES_99.0']:,.2f}",
         "ES_99.0_Pct_Portfolio": f"${(metrics_fo['ES_99.0'] / total_portfolio_ead)*100:.2f}%",
         "Model_Status": "REJECTED_SEVERE_UNDERESTIMATION"},
        {"Model_Specification": "2. Unsmoothed Vol + 90D Horizon (No Jumps)",
         "VaR_97.5_Dollar": f"${metrics_mid['VaR_97.5']:,.2f}",
         "ES_97.5_Dollar": f"${metrics_mid['ES_97.5']:,.2f}",
         "ES_99.0_Dollar": f"${metrics_mid['ES_99.0']:,.2f}",
         "ES_99.0_Pct_Portfolio": f"${(metrics_mid['ES_99.0'] / total_portfolio_ead)*100:.2f}%",
         "Model_Status": "WARNING_MISSING_JUMP_TAILS"},
        {"Model_Specification": "3. Validator Champion (90D L-VaR/L-ES + Merton Jumps)",
         "VaR_97.5_Dollar": f"${metrics_champ['VaR_97.5']:,.2f}",
         "ES_97.5_Dollar": f"${metrics_champ['ES_97.5']:,.2f}",
         "ES_99.0_Dollar": f"${metrics_champ['ES_99.0']:,.2f}",
         "ES_99.0_Pct_Portfolio": f"${(metrics_champ['ES_99.0'] / total_portfolio_ead)*100:.2f}%",
         "Model_Status": "APPROVED_CHAMPION"}
    ]

    return pd.DataFrame(comparison)

# --- Example Production Demonstration ---
if __name__ == "__main__":
  np.random.seed(42)
  n_loans = 10

  # Simulate a $1.2 Billion Private Credit Portfolio
  portfolio_data = pd.DataFrame({
      'Obligor_ID': [f"PC_Obligor_{i:02d}" for i in range(1, n_loans + 1)],
      'EAD': [150_000_000, 120_000_000, 200_000_000, 80_000_000, 110_000_000,
              95_000_000, 130_000_000, 140_000_000, 75_000_000, 100_000_000],
      'LGD': [0.35, 0.40, 0.45, 0.30, 0.50, 0.38, 0.42, 0.35, 0.48, 0.40],
      'Observed_Ann_Vol': np.random.uniform(0.04, 0.07, n_loans),         # Stale quarterly mark vol (4%-7%)
      'Smoothing_Phi': np.random.uniform(0.50, 0.65, n_loans),            # Heavy AR(1) Smoothing
      'Jump_Intensity_Lambda': np.random.uniform(0.15, 0.35, n_loans),    # 0.15 - 0.35 jumps per year
      'Jump_Mean_MuJ': np.random.uniform(-0.25, -0.15, n_loans),          # Downward jump mean (-15% to -25%)
      'Jump_Std_SigmaJ': np.random.uniform(0.08, 0.12, n_loans),          # Jump dispersion
      'Default_Barrier_K': np.random.uniform(0.65, 0.75, n_loans),        # Structural default barrier
      'Liquidity_Haircut_Eta': np.random.uniform(0.03, 0.06, n_loans)     # Liquidation friction
  })

  # Generate Equi-correlated Factor Covariance Matrix (rho = 0.30)
  rho = 0.30
  corr_mat = np.full((n_loans, n_loans), rho)
  np.fill_diagonal(corr_mat, 1.0)

  # Initialize Engine
  validator = IlliquidPrivateCreditVaRValidator(portfolio_data, corr_mat)

  # Print Unsmoothing Demonstration
  portfolio_data['Unsmoothed_Ann_Vol'] = validator.unsmooth_volatility()
  print("=== Step 1: AR(1) Volatility Unsmoothing Audit ===")
  print(portfolio_data[['Obligor_ID', 'Observed_Ann_Vol', 'Smoothing_Phi', 'Unsmoothed_Ann_Vol']].head(5).to_string(index=False))

  # Run Full Model Challenge Suite
  print("\n === Step 2: MRM Model Challenge - VaR & Expected Shortfall Comparison ===")
  mrm_report = validator.run_mrm_model_challenge_suite(n_sims=30000)
  print(mrm_report.to_string(index=False))








=== Step 1: AR(1) Volatility Unsmoothing Audit ===
   Obligor_ID  Observed_Ann_Vol  Smoothing_Phi  Unsmoothed_Ann_Vol
PC_Obligor_01          0.051236       0.503088            0.089111
PC_Obligor_02          0.068521       0.645486            0.147624
PC_Obligor_03          0.061960       0.624866            0.128951
PC_Obligor_04          0.057960       0.531851            0.104844
PC_Obligor_05          0.044681       0.527274            0.080310

 === Step 2: MRM Model Challenge - VaR & Expected Shortfall Comparison ===
                                  Model_Specification VaR_97.5_Dollar  ES_97.5_Dollar  ES_99.0_Dollar ES_99.0_Pct_Portfolio                    Model_Status
 1. Front-Office Baseline (10D, Smooth Vol, No Jumps)  $22,142,820.99  $24,528,875.16  $26,601,980.82                $2.22% REJECTED_SEVERE_UNDERESTIMATION
           2. Unsmoothed Vol + 90D Horizon (No Jumps)  $99,428,140.48 $113,131,820.76 $124,944,288.48               $10.41%      WARNING_MISSING_JUMP_TAILS
3. 